In [ ]:
import pandas as pd

In [ ]:
training_data = pd.read_csv('data/v0.2/flextrack-2025-training-data-v0.2.csv', parse_dates=['Timestamp_Local'])
test_data = pd.read_csv('data/v0.2/flextrack-2025-public-test-data-v0.2-with-flags.csv', parse_dates=['Timestamp_Local'])
random_preds = pd.read_csv('data/v0.2/flextrack-2025-random-prediction-data-v0.2.csv')

In [ ]:
time_col = "Timestamp_Local"
site_col = "Site"
index_cols = [site_col, time_col]
cls_features = ["Dry_Bulb_Temperature_C", "Global_Horizontal_Radiation_W/m2", "Building_Power_kW"]
reg_features = cls_features + ['Demand_Response_Flag']
target_col_classification = "Demand_Response_Capacity_kW"
target_cols = [target_col_classification]
# target_cols = ["Demand_Response_Flag", "Demand_Response_Capacity_kW"]

### Create Data Splits

In [ ]:
X_train = training_data[index_cols + reg_features].copy()
y_train = training_data[index_cols + target_cols].copy()

### Create lags and rolling features

In [ ]:
time_offsets = {"30min": 2, "1h": 4, "2h": 8, "6h": 4 * 6, "12h": 4 * 12, "1d": 24 * 4, "7d": 24 * 4 * 7, "m": 24 * 4 * 30, "y": 24 * 4 * 365}

In [ ]:
def get_lags_features(df, lags: list = ["1h","1d","7d"], diffs: list = [1, 4, 8]):
    stats = ['mean', 'std', 'min', 'max', 'median']
    lags_features = []
    grouped_df = df.groupby('Site')
    
    shifts = []
    for offset in lags:
        shift = df.set_index('Timestamp_Local').shift(freq=offset).reset_index().set_index(['Site',"Timestamp_Local"])
        shift.rename(columns=lambda x: f"{x}_shift_{offset}", inplace=True)
        shifts.append(shift)  

    windows_stats = []
    for window in lags:
        window_stats = []
        for stat in stats:
            window_stat = grouped_df.apply(
                lambda x: x.set_index('Timestamp_Local').rolling(window, min_periods=time_offsets[window]).agg(stat),
                include_groups=False
            ).reset_index().set_index(['Site',"Timestamp_Local"])
            window_stat.rename(columns=lambda x: f"{x}_roll_{window}_{stat}", inplace=True)
            window_stats.append(window_stat)
        windows_stats.append(pd.concat(window_stats, axis=1))
        
    diffs_dfs = []
    for diff in diffs:
        diffs_df = grouped_df.apply(
            lambda x: x.set_index('Timestamp_Local').diff(diff),
            include_groups=False
        ).reset_index().set_index(['Site',"Timestamp_Local"])
        diffs_df.rename(columns=lambda x: f"{x}_diff_{diff}", inplace=True)
        diffs_dfs.append(diffs_df)

        pct_change_df = grouped_df.apply(
            lambda x: x.set_index('Timestamp_Local').pct_change(diff),
            include_groups=False
        ).reset_index().set_index(['Site',"Timestamp_Local"])
        pct_change_df.rename(columns=lambda x: f"{x}_pct_change_{diff}", inplace=True)
        diffs_dfs.append(pct_change_df)

        # lags_features.append(pd.concat([*shifts, *window_stats, *diffs_dfs], axis=1))
    lags_features = pd.concat([*shifts, *windows_stats, *diffs_dfs], axis=1)
    return lags_features

In [ ]:
import numpy as np

def add_time_features(df, time_col='Timestamp_Local'):
    df['Minute'] = df[time_col].dt.minute
    df['Hour'] = df[time_col].dt.hour
    df['Day'] = df[time_col].dt.day
    df['Month'] = df[time_col].dt.month
    df['Year'] = df[time_col].dt.year
    df['Day_Of_Week'] = df[time_col].dt.dayofweek
    df['Is_Weekend'] = df['Day_Of_Week'].isin([5, 6]).astype(int)
    
    # Enhanced cyclical features
    df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
    df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
    df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
    df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
    df['Day_Of_Week_sin'] = np.sin(2 * np.pi * df['Day_Of_Week'] / 7)
    df['Day_Of_Week_cos'] = np.cos(2 * np.pi * df['Day_Of_Week'] / 7)
    
    # Hour categories for non-linear patterns
    df['Hour_Category'] = pd.cut(df['Hour'], bins=[-1, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'])
    
    # Season definition
    df['Season'] = df['Month'].map({12:0, 1:0, 2:0, 3:1, 4:1, 5:1, 6:2, 7:2, 8:2, 9:3, 10:3, 11:3})
    return df

In [ ]:
def create_features(df):
    df['Temp_by_Radiation'] = df['Dry_Bulb_Temperature_C'] * df['Global_Horizontal_Radiation_W/m2']
    df['Radiation_div_Temp'] = df['Global_Horizontal_Radiation_W/m2'] / df['Dry_Bulb_Temperature_C'] 
    df['Power_per_Temp'] = df['Building_Power_kW'] / (df['Dry_Bulb_Temperature_C'])
    return df

In [9]:
selected_lags = ["30min","1h", "2h", "6h", "12h", "1d", "7d"]
lags_features = get_lags_features(X_train, selected_lags)

In [10]:
site_stats = X_train.groupby(site_col)[cls_features].agg(['mean', 'std', "median"])
site_stats.columns = [f"{col}_{stat}" for col, stat in site_stats.columns.values]
X_train = X_train.merge(site_stats, left_on=site_col, right_index=True)

In [11]:
X_train = create_features(X_train)

In [12]:
# X_train = pd.concat([X_train.set_index(index_cols), *lags], axis=1).sort_index().reset_index()
X_train = X_train.set_index(index_cols).join(lags_features).reset_index()

In [13]:
X_train = add_time_features(X_train)

In [14]:
X_train = X_train.set_index(index_cols)

In [15]:
y_train = y_train.set_index(index_cols)
y_train = y_train.loc[X_train.index]

In [16]:
extra_time_festures = ['Hour', 'Month', 'Year', 'Day_Of_Week']

### Train CatBoost Regressor
Implement CatBoost regression using existing X_train and y_train for Demand_Response_Capacity_kW, evaluate with RMSE/MAE/R2, show feature importances, and save the model.

In [17]:
# Install catboost if not available (uncomment if running in a fresh environment)
# %pip install catboost

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import joblib
import os, json

TARGET = "Demand_Response_Capacity_kW"

# Align and drop rows with missing target
train_df = X_train.join(y_train[[TARGET]])
train_df = train_df.dropna(subset=[TARGET])

# Time-based split (last 20% per site)
site_splits = []
for site, grp in train_df.groupby(level=0):
    n = len(grp)
    cut = int(n * 0.8)
    site_splits.append((grp.iloc[:cut], grp.iloc[cut:]))
train_part = pd.concat([p[0] for p in site_splits])
valid_part = pd.concat([p[1] for p in site_splits])

feature_cols = [c for c in train_part.columns if c not in [TARGET]]

X_tr = train_part[feature_cols]
y_tr = train_part[TARGET]
X_va = valid_part[feature_cols]
y_va = valid_part[TARGET]

cat_features = ["Is_Weekend", "Hour_Category", "Season"]
train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
valid_pool = Pool(X_va, y_va, cat_features=cat_features)

model = CatBoostRegressor(
    loss_function='MAE',
    eval_metric='MAE',
    iterations=3000,
    random_seed=321,
    verbose=100,
    l2_leaf_reg=4,
    learning_rate=0.005,
    early_stopping_rounds=400,
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

va_preds = model.predict(X_va)

rmse = mean_squared_error(y_va, va_preds)
mae = mean_absolute_error(y_va, va_preds)
r2 = r2_score(y_va, va_preds)
print('RMSE:', rmse)
print('MAE:', mae)
print('R2:', r2)

# Top feature importances
importances = model.get_feature_importance(train_pool)
feat_imp = pd.DataFrame({'feature': feature_cols, 'importance': importances}).sort_values('importance', ascending=False).head(40)
print(feat_imp)

os.makedirs('models', exist_ok=True)
model_path = 'models/catboost_regressor_capacity.cbm'
model.save_model(model_path)
print(f'Model saved to {model_path}')

# Persist with joblib
joblib.dump(model, 'models/catboost_regressor_capacity.joblib')
print('Saved joblib model.')


0:	learn: 0.3440944	test: 0.4632338	best: 0.4632338 (0)	total: 88.1ms	remaining: 4m 24s
100:	learn: 0.3066710	test: 0.3988374	best: 0.3988374 (100)	total: 2.73s	remaining: 1m 18s
200:	learn: 0.2930477	test: 0.3752999	best: 0.3752999 (200)	total: 5.19s	remaining: 1m 12s
300:	learn: 0.2757410	test: 0.3436740	best: 0.3436740 (300)	total: 7.45s	remaining: 1m 6s
400:	learn: 0.2574951	test: 0.3076332	best: 0.3076332 (400)	total: 9.6s	remaining: 1m 2s
500:	learn: 0.2442441	test: 0.2794822	best: 0.2794685 (495)	total: 11.9s	remaining: 59.3s
600:	learn: 0.2369376	test: 0.2685574	best: 0.2685574 (600)	total: 14.3s	remaining: 57.1s
700:	learn: 0.2322348	test: 0.2628263	best: 0.2628263 (700)	total: 16.5s	remaining: 54.1s
800:	learn: 0.2276826	test: 0.2565045	best: 0.2565045 (800)	total: 18.7s	remaining: 51.4s
900:	learn: 0.2232984	test: 0.2506884	best: 0.2506884 (900)	total: 20.9s	remaining: 48.7s
1000:	learn: 0.2196295	test: 0.2463001	best: 0.2463001 (1000)	total: 23.1s	remaining: 46.1s
1100:	lea

In [18]:
def nmae(y_true, y_pred, building_power):
    return (np.mean(np.abs(y_true - y_pred)) / (np.mean(building_power) + 1e-9)) * 100

In [19]:
valid_part['Demand_Response_Capacity_kW_pred'] = va_preds

In [20]:
valid_part.reset_index().groupby('Site').apply(
    lambda x: nmae(x['Demand_Response_Capacity_kW'], x['Demand_Response_Capacity_kW_pred'], x['Building_Power_kW'])
)

/tmp/ipykernel_4750/3898361415.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  valid_part.reset_index().groupby('Site').apply(


Site
siteA    0.456618
siteB    1.797226
siteC    0.376057
dtype: float64

### Test Predictions (Capacity Regressor)
Generate test predictions using the CatBoost Regressor trained for Demand_Response_Capacity_kW.

In [21]:
X_test = test_data[index_cols + reg_features].copy()

In [22]:
lags_features = get_lags_features(X_test, selected_lags)

In [23]:
X_test = X_test.set_index(index_cols).join(lags_features).reset_index()

In [24]:
X_test = add_time_features(X_test)

In [25]:
from catboost import CatBoostRegressor, Pool
import os, numpy as np

# Load trained regressor
reg_model_path = 'models/catboost_regressor_capacity.cbm'
assert os.path.exists(reg_model_path), 'Regressor model file not found.'
reg = CatBoostRegressor()
reg.load_model(reg_model_path)

# Prepare and align test features to the model
X_test_feat = X_test.set_index(index_cols)
model_features = reg.feature_names_
X_test_feat_aligned = X_test_feat.drop(columns=[c for c in X_test_feat.columns if c not in model_features], errors='ignore')
for f in model_features:
    if f not in X_test_feat_aligned.columns:
        X_test_feat_aligned[f] = float('nan')
X_test_feat_aligned = X_test_feat_aligned[model_features]
cat_features = ["Is_Weekend", "Hour_Category", "Season"]
test_pool = Pool(X_test_feat_aligned, cat_features=cat_features)

# Predict capacity (clip to non-negative)
cap_preds = reg.predict(test_pool)
# cap_preds = np.clip(cap_preds, 0, None)

pred_out = X_test_feat_aligned.reset_index()[index_cols+['Demand_Response_Flag']].copy()
pred_out['Demand_Response_Capacity_kW'] = cap_preds

pred_out.head()

,Site,Timestamp_Local,Demand_Response_Flag,Demand_Response_Capacity_kW
0,siteD,2020-01-01 00:00:00,0.0,0.489044
1,siteD,2020-01-01 00:15:00,0.0,0.131963
2,siteD,2020-01-01 00:30:00,0.0,0.123529
3,siteD,2020-01-01 00:45:00,0.0,0.034018
4,siteD,2020-01-01 01:00:00,0.0,0.018944


In [26]:
import os
import time
os.makedirs('data/submissions/main-phase', exist_ok=True)
submission_path = f'data/submissions/main-phase/{time.strftime("%Y%m%d-%H%M%S")}.csv'
pred_out.to_csv(submission_path, index=False)
print(f'Saved capacity predictions to {submission_path}')

Saved capacity predictions to data/submissions/main-phase/20250923-110309.csv
